In [1]:
# 有向图的邻接表表示
class AbstractCollection(object):
    """所有集合类型的抽象集合"""
    def __init__(self,sourceCollection):
        """构造一个抽象集合的集合，如果sourceCollection中存在元素，则复制之"""
        self._size=0        #命名约定，单下划线开始的成员变量叫作保护变量，只有类实例和子类实例能访问，不能用from module import导入
        if sourceCollection:
            for item in sourceCollection:
                self.add(item)
    def __len__(self):
        """返回元素数量"""
        return self._size
    def isEmpty(self):
        return len(self)==0 #return True if len(self)==0
    def __str__(self):
        return "["+",".join(map(str,self))+"]" #在用户交互应用中用到
    def __add__(self,other):
        """合并self和other并返回一个新的集合"""
        result=type(self)(self)
        for item in other:
            result.add(item)
        return result
    def __eq__(self,other):
        """两个类对象即使各方面数值完全相同，也不会相等，因为在内存中不同位置，__eq__让它们相等"""
        if self is other:
            return True
        if type(self)!=type(other):
            return False
        if len(self)!=len(other):
            return False
        otherItems=iter(other)
        for item in self:
            if item!=next(otherItems):
                return False
        return True
class LinkedEdge(object):
    #边含有一个起始点，一个终点，权重和标记
    def __init__(self,fromVertex,toVertex,weight=None):
        self._vertex1=fromVertex
        self._vertex2=toVertex
        self._weight=weight
        self._mark=False
    def clearMark(self):
        self._mark=False
    def __eq__(self,other):
        """两个类对象即使各方面数值完全相同，也不会相等，因为在内存中不同位置，
        __eq__让它们相等"""
        if self is other:
            return True
        if type(self)!=type(other):
            return False
        return self._vertex1==other._vertex1 and self._vertex2==other._vertex2
    def getOtherVertex(self,thisVertex):
        """输入边的一个顶点，返回另一个顶点"""
        if thisVertex==None or thisVertex==self._vertex2:
            return self._vertex1
        else:
            return self._vertex2
    def getToVertex(self):
        """获取到达顶点"""
        return self._vertex2
    def getWeight(self):
        return self._weight
    def isMarked(self):
        """判断边是否被标记"""
        return self._mark
    def setMark(self):
        self._mark=True
    def setWeight(self,weight):
        self._weight=weight
    def __str__(self):
        return str(self._vertex1)+">"+\
    str(self._vertex2)+":"+\
    str(self._weight)
class LinkedVertex(object):
    #顶点含有一个标签，相连边的列表和标记
    def __init__(self,label):
        self._label=label
        self._edgeList=list()
        self._mark=False
    def clearMark(self):
        self._mark=False
    def getLabel(self):
        return self._label
    def isMarked(self):
        return self._mark
    def setLabel(self,label,g):    #g是graph图，图的顶点字典pop
        g._vertices.pop(self._label,None) #如果键self._label存在，返回其值并删除项目，否则返回None
        g._vertices[label]=self #重新将键label指向值self，操作图的实例
        self._label=label #操作顶点的实例
    def setMark(self):
        self._mark=True
    def __str__(self):
        return str(self._label)
    def __eq__(self,other):
        if self is other:
            return True
        if type(self)!=type(other):
            return False
        return self.getLabel()==other.getLabel()
    #用于邻接图的方法
    def addEdgeTo(self,toVertex,weight):
        """为self顶点添加一条权重为weight的到toVertex的边"""
        edge=LinkedEdge(self,toVertex,weight)
        self._edgeList.append(edge)
    def getEdgeTo(self,toVertex):
        """若顶点关联边中存在到toVertex的边，返回该边，否则返回None"""
        edge=LinkedEdge(self,toVertex)
        try:
            return self._edgeList[self._edgeList.index(edge)] #list.index()返回查找对象的索引位置
        except:
            return None
    def incidentEdges(self):
        """返回此顶点的关联边的迭代器"""
        return iter(self._edgeList)
    def neighboringVertices(self):
        """返回此顶点的相邻顶点"""
        vertices=list()
        for edge in self._edgeList:
            vertices.append(edge.getOtherVertex(self)) #获取此顶点相连的另一顶点，无论出还是入
        return iter(vertices)
    def removeEdgeTo(self,toVertex):
        edge=LinkedEdge(self,toVertex)
        if edge in self._edgeList:
            self._edgeList.remove(edge)
            return True
        else:
            return False
class LinkedDirectedGraph(AbstractCollection):
    #图有顶点计数，边计数，一个标签/顶点配对的字典
    def __init__(self,sourceCollection=None):
        self._edgeCount=0
        self._vertices={} #键为label，值为LinkedVertex实例，包括label、相邻边和标记
        AbstractCollection.__init__(self,sourceCollection)
    #用于清除、标记、返回大小和字符表示的方法
    def clear(self):
        """清除图中所有顶点"""
        self._size=0
        self._edgeCount=0
        self._vertices={}
    def clearEdgeMarks(self):
        for edge in self.edges():
            edge.clearMark()
    def clearVertexMarks(self):
        for vertex in self.vertices():
            vertex.clearMark()
    def sizeEdges(self):
        """返回图中边的数量"""
        return self._edgeCount
    def sizeVertices(self):
        """返回图中顶点的数量"""
        return len(self)
    def __str__(self):
        result = str(self.sizeVertices())+"Vertices:"
        for vertex in self._vertices: #vertex为键label
            result+=" "+str(vertex)
        result+="\n"; #换行符
        result+=str(self.sizeEdges())+"Edges:"
        for edge in self.edges():
            result+=" "+str(edge)
        return result
    def add(self,label):
        """与其他集合兼容"""
        self.addVertex(label)
    #与顶点相关的方法
    def addVertex(self,label):
        if self.containsVertex(label):
            print(label,"添加失败，已存在相同标签的顶点。")
        else:
            self._vertices[label]=LinkedVertex(label) #以label为键的字典表示顶点，值为一个LinkedVerte实例
            self._size+=1
    def containsVertex(self,label):
        return label in self._vertices #return True if label in self._vertices
    def getVertex(self,label):
        return self._vertices[label]
    def removeVertex(self,label):
        removedVertex=self._vertices.pop(label,None)
        if removedVertex is None:
            return False
        #检验所有其他顶点，以移除指向被移除顶点的边
        for vertex in self.vertices():
            if vertex.removeEdgeTo(removedVertex):
                self._edgeCount-=1
        #检验所有从被移除顶点发出的边
        for edge in removedVertex.incidentEdges():
            self._edgeCount-=1
        self._size-=1
        return True
    #与边相关的方法
    def addEdge(self,fromLabel,toLabel,weight):
        #先获取两个标签指向的顶点
        fromVertex=self.getVertex(fromLabel)
        toVertex=self.getVertex(toLabel)
        if fromVertex.getEdgeTo(toVertex)==None:
            #调用LinkedVertex类的方法添加边
            fromVertex.addEdgeTo(toVertex,weight)
            #调用图的方法
            self._edgeCount+=1
        else:
            print("添加失败，已存在一条从",fromVertex,"出发连向",toVertex,"的边。")
    def containsEdge(self,fromLabel,toLabel):
        return self.getEdge(fromLabel,toLabel)!=None
    def getEdge(self,fromLabel,toLabel):
        fromVertex=self.getVertex(fromLabel)
        toVertex=self.getVertex(toLabel)
        return fromVertex.getEdgeTo(toVertex)
    def removeEdge(self,fromLabel,toLabel):
        fromVertex=self.getVertex(fromLabel)
        toVertex=self.getVertex(toLabel)
        edgeRemovedFlg=fromVertex.removeEdgeTo(toVertex)
        if edgeRemovedFlg:
            self._edgeCount-=1
        return edgeRemovedFlg
    #迭代器
    def __iter__(self):
        """默认图的迭代为按顶点迭代？"""
        return self.vertices()
    def edges(self):
        """支持按边迭代"""
        result=list()
        for vertex in self.vertices():
            result+=list(vertex.incidentEdges()) #self._vertices值的列表，每个值的_edgeList()列表
        return iter(result) #二维还是三维矩阵？二维，列表相加，结果等同于extend
    def vertices(self):
        """支持按顶点迭代"""
        return iter(self._vertices.values()) #返回字典self._vertices值的列表
    def incidentEdges(self,label):
        """支持按给出的顶点的关联边迭代"""
        return self.getVertex(label).incidentEdges()
    def neighboringVertices(self,label):
        """支持按给出的顶点的相邻顶点迭代"""
        return self.getVertex(label).neighboringVertices()
#Example use of the graph collection
"""
g=LinkedDirectedGraph()
#插入顶点
g.addVertex("A")
g.addVertex("B")
g.addVertex("C")
g.addVertex("D")
g.addVertex("E")
#插入边
g.addEdge("A","B",3)
g.addEdge("A","C",2)
g.addEdge("B","D",1)
g.addEdge("C","D",1)
g.addEdge("D","E",2)
print(g)
print("A顶点的相邻顶点：")
for vertex in g.neighboringVertices("A"):
    print(vertex)
print("A顶点的相连边：")
for edge in g.incidentEdges("A"):
    print(edge)"""
class GraphDemoModel(object):
    """创建图并运行算法"""
    def __init__(self):
        self._graph=None
        self._startLabel=None
    def createGraph(self,rep,startLabel):
        """从用户输入的rep和startLabel创建图"""
        self._graph=LinkedDirectedGraph()
        self._startLabel=startLabel
        edgeList=rep.split() #str.split(str=""分隔符,num=string.count(str)分割次数)，返回分割后的字符串列表
        for edge in edgeList:
            if not '>' in edge:
                #若无>，则将edge视作一个未连接的顶点
                if not self._graph.containsVertex(edge): #若edge作为顶点不存在于图中，则添加这一顶点
                    self._graph.addVertex(edge)
                else:
                    self._graph=None #若edge作为顶点已存在于图中，则创建错误，清除图
                    return "顶点重复"
            else:
                #有>则视作两个顶点和一条边
                bracketPos=edge.find('>') #str.find(str,beg=0,end=len(string))，若含有str则返回位置，否则返回-1
                colonPos=edge.find(':')
                if bracketPos==-1 or colonPos==-1 or bracketPos>colonPos:
                    self._graph=None
                    return " > 或者 : 存在问题"
                fromLabel=edge[:bracketPos]
                toLabel=edge[bracketPos+1:colonPos]
                weight=edge[colonPos+1:]
                if weight.isdigit(): #str.isdight()，如果字符串只包含数字返回True，否则False
                    weight=int(weight)
                if not self._graph.containsVertex(fromLabel):
                    self._graph.addVertex(fromLabel)
                if not self._graph.containsVertex(toLabel):
                    self._graph.addVertex(toLabel)
                if self._graph.containsEdge(fromLabel,toLabel):
                    self._graph=None
                    return "边重复"
                self._graph.addEdge(fromLabel,toLabel,weight)
        vertex=self._graph.getVertex(startLabel)
        if vertex is None:
            self._graph=None
            return "起始顶点不存在于图中"
        else:
            vertex.setMark()
            return "成功创建图"
    def getGraph(self):
        """返回图的字符串表示"""
        if not self._graph:
            return None
        else:
            return str(self._graph)
    def getStartLabel(self):
        return self._startLabel
    def run(self,algorithm):
        """在图上运行给定的算法"""
        if self._graph is None:
            return None
        else:
            return algorithm(self._graph,self._startLabel)
#以下为algorithm，包括遍历、最短路径、最小生成树和拓扑排序
"""def traverseAll(graph,startLabel):
    stack=LinkedStack()
    graph.clearVertexMarks()
    for vertex in graph.vertices():
        if not vertex.isMarked():
            dfs(graph,vertex,stack)
    return stack"""
def dfs(graph,startVertex,stack):
    startVertex.setMark()
    for vertex in startVertex.neighboringVertices():
        if not vertex.isMarked():
            dfs(graph,vertex,stack)
    stack.push(startVertex)
def topoSort(g,startLabel=None):
    stack=LinkedStack()
    g.clearVertexMarks()
    for v in g.vertices():
        if not v.isMarked():
            dfs(g,v,stack)
    return stack
"""使用heap堆获取最小值，无法实现
def spanTree(g,startLabel):
    默认为强连通有向图
    g.clearVertexMarks()
    g.clearEdgeMarks()
    stack=LinkedStack()
    tree=[]
    v=g.getVertex(startLabel)
    if not v.isMarked():
        v.setMark()
        tree.append(v)
        for edge in v.incidentEdges():
            stack.push(edge)
        k=1
        while k<g.sizeVertices():
            popedEdge=stack.pop()
            w=popedEdge.getOtherVertex(v)
            if not w.isMarked():
                w.setMark()
                tree.append(w)
                for edge2 in w.incidentEdges():
                    stack.push(edge2)
                k+=1
    return tree"""
def spanTree(g,startLabel):
    g.clearVertexMarks()
    g.clearEdgeMarks()
    v=g.getVertex(startLabel)
    v.setMark()
    markedVertices=list()
    markedVertices.append(v)
    minEdgeList=[]
    count=1
    while count<g._size:
        edgeList=[] #本来把edgeList放在while上面，结果每轮循环不清空，最小生成树里都是重复的最小权重边。我还以为是顶点Mark没打上，一直在改下面的setMark，改了半年才发现真正的问题所在。
        for markedVertex in markedVertices:
            for u in g.vertices():
                if (not u.isMarked()) and g.containsEdge(markedVertex._label,u._label):
                    w=g.getEdge(markedVertex._label,u._label)
                    edgeList.append(w)
        #minEdge=edgeList.findMinEdge()
        minWeight=float("inf")
        for edge in edgeList:
            if edge._weight<minWeight:
                minWeight=edge._weight
                minEdge=edge
        minEdgeList.append(minEdge)
        fromLabel=minEdge._vertex1._label #图根据字符串label查找顶点，如果不加._label则报错unhashable type
        toLabel=minEdge._vertex2._label
        g.getEdge(fromLabel,toLabel).setMark()
        g.getVertex(toLabel).setMark() #只有对图操作才能改变图中顶点和边的Mark
        markedVertices.append(g.getVertex(toLabel))
        count+=1
    return minEdgeList
def shortestPaths(g,startLabel):
    """包括初始化步骤和计算步骤，首先是初始化步骤"""
    results=[[]for i in range(g._size)]
    """results为一个有n行，3列的矩阵，n为顶点数，第一列为顶点标签，第二列为源点到该顶点距离，第三列为该顶点直接父母顶点"""
    included=["False" for i in range(g._size)]
    """included用n个布尔数（不懂换成字符串了）记录第n个顶点是否被计入最短路径中"""
    row=0
    for vertex in g.vertices():
        results[row].append(vertex._label)
        if vertex._label==startLabel:
            results[row].append(int(0))
            results[row].append(None)
            included[row]="True"
        elif g.containsEdge(startLabel,vertex._label):
            edge=g.getEdge(startLabel,vertex._label)
            results[row].append(edge._weight)
            results[row].append(startLabel)
            included[row]="False"
        else:
            results[row].append(float("inf"))
            results[row].append(None)
            included[row]="False"
        row+=1
    """以下为计算步骤"""
    row=0
    while True:
        minWeight=float("inf")
        for row in range(g._size):
            if included[row]=="False" and results[row][1]<minWeight:
                minWeight=results[row][1]
                minVertex=row
        included[minVertex]="True"
        newStartLabel=results[minVertex][0]
        row=0
        for row in range(g._size):
            if included[row]=="False" and g.containsEdge(newStartLabel,results[row][0]):
                newWeight=minWeight+g.getEdge(newStartLabel,results[row][0])._weight
                if newWeight<results[row][1]:
                    results[row][1]=newWeight
                    results[row][2]=newStartLabel
        if not "False" in included:
            break
    return results
"""def findMinEdge(edgeList):   本来想把最小生成树的一部分算法写成另一个功能函数，但是不在同一个类里，无法被调用
    minWeight=float("inf")
    for edge in edgeList:
        if edge._weight<minWeight:
            minWeight=edge._weight
            minEdge=edge
    return minEdge"""
class GraphDemoView(object):
    """与用户交互的视图界面"""
    def __init__(self):
        self._model=GraphDemoModel()
    def run(self):
        """以菜单驱动的应用控制"""
        menu="主菜单\n"+\
        "1 用键盘输入一个图\n"+\
        "2 从文件中输入一个图\n"+\
        "3 查看当前图\n"\
        "4 单源点最短路径\n"\
        "5 最小生成树\n"\
        "6 拓扑排序\n"\
        "7 退出程序\n"
        while True:
            command=self._getCommand(7,menu)
            if command==1:
                self._getFromKeyboard()
            elif command==2:
                self._getFromFile()
            elif command==3:
                print(self._model.getGraph())
            elif command==4:
                print("Paths:\n",self._model.run(shortestPaths))
            elif command==5:
                print("Tree:"," ".join(map(str,self._model.run(spanTree))))
            elif command==6:
                print("Sort:"," ".join(map(str,self._model.run(topoSort))))
            else:
                break
    def _getCommand(self,high,menu):
        """从用户获取并返回一个控制命令"""
        prompt="Enter a number [1-"+str(high)+"]:" #prompt提示符
        commandRange=list(map(str,range(1,high+1))) #map将1-high+1转换为字符，因为用户输入的是字符
        error="Error, number must be 1 to"+str(high)
        while True:
            print(menu)
            command=input(prompt)
            if command in commandRange:
                return int(command)
            else:
                print(error)
    def _getFromKeyboard(self):
        """用键盘输入一个图的字符描述并创建图"""
        rep=""
        while True:
            edge=input("输入一条边，回车结束：") #输入中不可包含空格
            if edge=="":
                break
            rep+=edge+" "
        startLabel=input("输入起始顶点的标签：")
        print(self._model.createGraph(rep, startLabel))
    def _getFromFile(self):
        """从文件中输入图的描述并创建图"""
        pass #暂先不写
#启动应用
#GraphDemoView().run() #为什么可以直接运行类？
#GraphDemoView-run-getCommand-1-getFromKeyboard-self._model-graphDemoModel-createGraph-LinkedDirectGraph
#链栈代码
class AbstractStack(AbstractCollection):
    def __init__(self,sourceCollection=None):
        AbstractCollection.__init__(self,sourceCollection)
    def add(self,item):
        self.push(item)
class Node(object):
    def __init__(self,data,next=None):
        self.data=data
        self.next=next
class LinkedStack(AbstractStack):
    def __init__(self,sourceCollection=None):
        self._items=None
        AbstractStack.__init__(self,sourceCollection)
    #访问方法
    def __iter__(self):
        def visitNodes(node):
            if not node is None:
                visitNodes(node.next)
                tempList.append(node.data)
        tempList=list()
        visitNodes(self._items)
        return iter(tempList)
    def peek(self):
        """返回栈顶元素"""
        if self.isEmpty():
            raise KeyError("栈为空。")
        return self._items.data
    #设置方法
    def clear(self):
        self._size=0
        self._items=None
    def push(self,item):
        """从栈顶添加元素"""
        self._items=Node(item,self._items)
        self._size+=1
    def pop(self):
        """从栈顶出栈"""
        if self.isEmpty():
            raise KeyError("栈为空。")
        data=self._items.data
        self._items=self._items.next
        self._size-=1
        return data
GraphDemoView().run()


主菜单
1 用键盘输入一个图
2 从文件中输入一个图
3 查看当前图
4 单源点最短路径
5 最小生成树
6 拓扑排序
7 退出程序



Enter a number [1-7]: 1


KeyboardInterrupt: Interrupted by user